# 🧠 MEGA — GNN + Deep RL for Misinformation Suppression
### Final-Year Computer Engineering Project — Kaggle GPU Run

**Optimised for Kaggle T4 (16 GB GPU / 30 GB RAM): target 7–9 h wall-clock (v7 — evaluation fixes).**

Key optimisations vs original:
- **GNN risk caching** — inference runs every `RISK_CACHE_STEPS` timesteps, not every step (~8× speedup)
- **Lighter GNN** — 3-layer residual GCN (hidden 64) instead of 4-layer + GAT (hidden 128); still strong
- **Smaller DQN** — hidden 128 instead of 256
- **Tuned episode schedule** — p2p 800, ca_grqc 1000, facebook 2200; 2 train seeds each
- **Degree centrality** replaces full betweenness for Facebook (saves ~15 min)
- **Facebook-specific fixes** — tighter budget cap (100), slower ε-decay (0.9975), shorter GNN lookahead (5 steps), density-scaled reward
- **FIX-G (v7)** — `evaluate()` uses same seed node as baselines (rank-3); prevents systematic bias from hub node mismatch
- **FIX-H (v7)** — suppression % computed against matched `none` baseline at agent eval seeds; honest apples-to-apples comparison
- **FIX-I (v7)** — `_vspread()` accepts `protected` node set; cured/muted nodes cannot be re-infected in the same timestep on dense graphs
- **Vectorised spread** via pre-built sparse adjacency matrix (unchanged — already fast)
- **float16 GNN features** on GPU to halve VRAM during forward pass
- **Aggressive GPU memory cleanup** between graphs

Run cells 0 → 9 in order. Checkpoints saved to `/kaggle/working/weights/`.


## Cell 0 — Environment Reset

In [ ]:
import os, glob, gc, warnings, time
warnings.filterwarnings('ignore')

print("MEGA_V6 — GNN+RL Misinformation Suppression (Kaggle GPU Optimised)")
for f in glob.glob('cc_*.npz') + glob.glob('spec_*.npz'):
    os.remove(f); print(f'  removed cache: {f}')
os.makedirs('weights', exist_ok=True)
os.makedirs('plots',   exist_ok=True)
gc.collect()
print('Ready — run cells 1 to 9 in order.')


## Cell 1 — Installs, Imports, Hyperparameters, Graph Loading

In [ ]:
# ─── Install torch-geometric — P100 (sm_60) compatible ───────────────────────
# ROOT CAUSE OF THE CRASH:
#   "CUDA error: no kernel image is available for execution on the device"
#   PyG prebuilt wheels for cu121 are compiled with sm_70/sm_75/sm_80+ only.
#   Kaggle's Tesla P100-PCIE-16GB is compute capability 6.0 (Pascal / sm_60).
#   Those wheels have NO kernel image for sm_60, so the very first CUDA op dies.
#
# FIX: install torch-scatter and torch-sparse from SOURCE so they compile for
#      the actual GPU on this machine. torch-geometric itself is pure Python.
#      torch-spline-conv is skipped (not needed for GCNConv).
#
import subprocess, sys, torch, os

torch_ver  = torch.__version__.split('+')[0]
cuda_avail = torch.cuda.is_available()

if cuda_avail:
    cc_major = torch.cuda.get_device_properties(0).major
    cc_minor = torch.cuda.get_device_properties(0).minor
    print(f'GPU compute capability: sm_{cc_major}{cc_minor}')
    # sm_60 = P100, sm_70 = V100, sm_75 = T4, sm_80 = A100
    # Prebuilt PyG wheels support sm_70+ only. Build from source for sm_60.
    need_source_build = (cc_major < 7)
else:
    need_source_build = False

print(f'torch={torch_ver}, cuda={cuda_avail}, source_build={need_source_build}')

# Skip reinstall if torch-geometric is already importable (saves ~25 min on re-runs)
def _pyg_installed():
    try:
        import torch_geometric, torch_scatter, torch_sparse
        return True
    except ImportError:
        return False

if _pyg_installed():
    print('torch-geometric already installed — skipping install step.')
elif need_source_build:
    # Install torch-geometric (pure Python — works as-is)
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch-geometric', '-q'], check=True)
    # Build torch-scatter and torch-sparse from source for sm_60
    # FORCE_CUDA=1 ensures CUDA extensions are compiled; TORCH_CUDA_ARCH_LIST pins sm_60
    env = os.environ.copy()
    env['FORCE_CUDA']          = '1'
    env['TORCH_CUDA_ARCH_LIST'] = '6.0'   # P100 exactly
    for pkg in ['torch-scatter', 'torch-sparse']:
        print(f'  Building {pkg} from source (sm_60) ...')
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', pkg,
             '--no-binary', pkg, '-q'],
            env=env, capture_output=True, text=True
        )
        if r.returncode != 0:
            print(f'  WARNING: {pkg} source build failed, falling back to CPU ops')
            print(r.stderr[-500:])
            subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
else:
    # V100/T4/A100+: use PyG's prebuilt wheel index (cu121 covers cu121-cu128 torch)
    _CUDA_WHEEL_MAP = {
        '128':'cu121','127':'cu121','126':'cu121','125':'cu121',
        '124':'cu121','123':'cu121','122':'cu121','121':'cu121','118':'cu118',
    }
    raw = 'cu' + torch.version.cuda.replace('.','') if cuda_avail else 'cpu'
    tag = _CUDA_WHEEL_MAP.get(raw.replace('cu',''), raw)
    url = f'https://data.pyg.org/whl/torch-{torch_ver}+{tag}.html'
    print(f'  Using prebuilt wheels: {url}')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install',
         'torch-geometric', 'torch-scatter', 'torch-sparse',
         '-q', '--find-links', url],
        check=True
    )

print('torch-geometric ready.')

# ─── Validate CUDA works before proceeding ────────────────────────────────────
if cuda_avail:
    import torch_geometric
    import torch
    # Tiny smoke-test: create a 1-hop GCN tensor op on GPU
    _x  = torch.ones(3, 4, device='cuda')
    _ei = torch.tensor([[0,1],[1,2]], dtype=torch.long, device='cuda')
    # If this doesn't crash, the CUDA kernels are compatible
    from torch_geometric.utils import add_self_loops
    _ei2, _ = add_self_loops(_ei, num_nodes=3)
    del _x, _ei, _ei2
    torch.cuda.empty_cache()
    print('CUDA smoke-test passed.')

# ─── Imports ──────────────────────────────────────────────────────────────────
import networkx as nx, numpy as np, random, gc
import urllib.request, gzip, scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — safe on Kaggle
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
from collections import defaultdict
import torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv
import copy, json, datetime, shutil, glob, time
from IPython.display import FileLink, display
print('Imports OK')

# ─── Global hyperparameters ───────────────────────────────────────────────────
SEED_VALUE       = 42
TIMESTEPS        = 50
CRED             = 0.85
NUM_EVAL_SEEDS   = 7
MOD_DELAY        = 5
SEED_NODE_RANK   = 3
SPEC_DIM         = 8
GNN_HIDDEN       = 64       # lean GNN — fast on P100
DQN_HIDDEN       = 128
PER_CAP          = 15_000   # reduced from 20k to save RAM (~70 MB saved)
RISK_CACHE_STEPS = 4        # fresher GNN guidance — better Q-targets on sparse graphs
PRE_SPREAD_STEPS = 2        # 2-step pre-spread: epidemic is live but agent still has meaningful room to act

# Episode schedule tuned for P100 (slightly fewer than T4 to hit 5-6 h):
EPISODE_SCHEDULE = {
    'p2p_gnutella': 1200,   # increased: sparse topology needs more episodes to converge
    'ca_grqc':      1200,   # increased: extra 200 episodes for robustness
    'facebook':     2500,   # increased: dense graph benefits from longer training
}
GNN_UPDATE_FREQ  = {'p2p_gnutella': 3, 'ca_grqc': 3, 'facebook': 2}
BUDGET_CAPS      = {'p2p_gnutella': 200, 'ca_grqc': 80, 'facebook': 100}  # tighter cap: makes budget signal informative + forces prioritisation
NUM_TRAIN_SEEDS  = 3

np.random.seed(SEED_VALUE); random.seed(SEED_VALUE); torch.manual_seed(SEED_VALUE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'  GPU: {props.name}  VRAM: {props.total_memory // 1024**2} MB')
    torch.backends.cudnn.benchmark = True

# ─── Graph loading ────────────────────────────────────────────────────────────
DATASETS = {
    'p2p_gnutella': ('https://snap.stanford.edu/data/p2p-Gnutella08.txt.gz',    'p2p_gnutella.txt'),
    'ca_grqc':      ('https://snap.stanford.edu/data/ca-GrQc.txt.gz',           'ca_grqc.txt'),
    'facebook':     ('https://snap.stanford.edu/data/facebook_combined.txt.gz', 'facebook.txt'),
}
graphs = {}
for name, (url, txt) in DATASETS.items():
    if not os.path.exists(txt):
        print(f'Downloading {name} ...')
        gz = txt + '.gz'
        urllib.request.urlretrieve(url, gz)
        with gzip.open(gz, 'rb') as fi, open(txt, 'wb') as fo:
            fo.write(fi.read())
        os.remove(gz)
    G = nx.read_edgelist(txt, nodetype=int, comments='#')
    G = nx.convert_node_labels_to_integers(G)
    if not nx.is_connected(G):
        G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
        G = nx.convert_node_labels_to_integers(G)
    graphs[name] = G
    avg_deg = 2 * G.number_of_edges() / G.number_of_nodes()
    print(f'  {name}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, avg_deg={avg_deg:.1f}')

GRAPH_ORDER = ['p2p_gnutella', 'ca_grqc', 'facebook']
print('\nGraphs ready.')


## Cell 2 — Spectral Embedding (structural node features)

Top-8 Laplacian eigenvectors per graph, cached to disk. Gives the GNN
positional/structural awareness — especially useful for Facebook's dense
community structure. Uses `eigsh` (sparse solver) so runs in ~1–3 min per graph.


In [ ]:
def spectral_embedding(G, k=SPEC_DIM, graph_seed=0):
    """Top-k eigenvectors of the normalised graph Laplacian (cached)."""
    n     = G.number_of_nodes()
    cpath = f'spec_{n}_{graph_seed}.npz'
    if os.path.exists(cpath):
        arr = np.load(cpath)['emb']
        print(f'  spectral cache hit: {cpath}  shape={arr.shape}')
        return arr

    t0 = time.time()
    L  = nx.normalized_laplacian_matrix(G, nodelist=range(n)).astype(np.float32)
    k_ = min(k + 1, n - 2)
    vals, vecs = spla.eigsh(L, k=k_, which='SM')
    order = np.argsort(vals)
    vecs  = vecs[:, order]
    emb   = vecs[:, 1:k+1] if vecs.shape[1] > k else vecs[:, 1:]
    if emb.shape[1] < k:
        emb = np.pad(emb, ((0, 0), (0, k - emb.shape[1])))
    emb = emb.astype(np.float32)
    np.savez(cpath, emb=emb)
    print(f'  {n} nodes: spectral computed in {time.time()-t0:.1f}s  shape={emb.shape}')
    return emb

spectral = {}
for name, G in graphs.items():
    print(f'Computing spectral embedding for {name} ...')
    spectral[name] = spectral_embedding(G)

print('\nSpectral embeddings ready.')


## Cell 3 — Simulator

In [ ]:
class Sim:
    def __init__(self, graph, graph_seed=0):
        n   = len(graph.nodes())
        rng = np.random.default_rng(graph_seed)
        self.n            = n
        self.belief       = rng.uniform(0.0, 0.02, n).astype(np.float32)
        self.influence    = rng.beta(4, 2, n).astype(np.float32)
        self.skepticism   = rng.beta(1.5, 5.5, n).astype(np.float32)
        self.share_prob   = rng.uniform(0.65, 0.95, n).astype(np.float32)
        self.trust_score  = rng.uniform(0.8, 1.0, n).astype(np.float32)
        self._orig_inf    = self.influence.copy()
        self._orig_bel    = self.belief.copy()
        self.neighbors    = [list(graph.neighbors(v)) for v in range(n)]

    def reset(self, seed_id, episode_seed=0, pre_spread=0):
        self.belief[:]    = self._orig_bel.copy()
        self.influence[:] = self._orig_inf.copy()
        ep_rng = np.random.default_rng(episode_seed)
        self.belief += ep_rng.uniform(0.0, 0.005, self.n).astype(np.float32)
        self.belief  = np.clip(self.belief, 0.0, 1.0)
        self.belief[seed_id] = 1.0
        frontier = [seed_id]
        # Pre-spread: let epidemic run uncontrolled for N steps so agent always
        # faces a live spreading epidemic (prevents trivial t=0 single-node cure exploit)
        for _ in range(pre_spread):
            if not frontier: break
            # Simple explicit spread using neighbors list
            nxt = []
            for src in frontier:
                for nbr in self.neighbors[src]:
                    if self.belief[nbr] <= 0.5:
                        d = (self.influence[src] * 0.85 * 1.5 * self.share_prob[src]) / (1.0 + self.skepticism[nbr]) * self.trust_score[nbr]
                        if d > 0.03:
                            old_b = self.belief[nbr]
                            self.belief[nbr] = min(1.0, self.belief[nbr] + d)
                            if old_b <= 0.5 and self.belief[nbr] > 0.5:
                                nxt.append(nbr)
            frontier = list(set(nxt))
        return list(np.where(self.belief > 0.5)[0])

print('Sim ready.')


## Cell 4 — Baselines + Budget

In [ ]:
GRAPH_SEED = 0

def budget(n, gname=None):
    cap = BUDGET_CAPS.get(gname, None)
    raw = max(15, int(n * 0.06))
    return min(raw, cap) if cap else raw

def run_baseline(G, seed_id, strategy='none', episode_seed=SEED_VALUE, gname=None):
    sim        = Sim(G, graph_seed=GRAPH_SEED)
    active     = sim.reset(seed_id, episode_seed=episode_seed, pre_spread=PRE_SPREAD_STEPS)
    b          = budget(sim.n, gname)
    spread_rng = np.random.default_rng(episode_seed + 999)
    deg_order  = sorted(range(sim.n), key=lambda v: len(sim.neighbors[v]), reverse=True)
    history    = [1]; done = False

    for t in range(TIMESTEPS):
        if not active: break
        if strategy in ('active_degree', 'active_random') and t > 0:
            infected = np.where(sim.belief > 0.5)[0]
            if len(infected) > 0:
                if strategy == 'active_degree':
                    targets = sorted(infected, key=lambda v: len(sim.neighbors[v]),
                                     reverse=True)[:b]
                else:
                    targets = spread_rng.choice(infected, size=min(b, len(infected)), replace=False)
                for v in targets:
                    sim.belief[v]    = max(0.0, sim.belief[v] - 0.85)
                    sim.influence[v] = sim.influence[v] * 0.3   # also dampen influence
        elif not done and t == MOD_DELAY:
            if strategy == 'passive_degree':
                for v in deg_order[:b]:
                    sim.influence[v] = 0.0
            done = True

        nxt = []
        for src in active:
            if sim.influence[src] == 0.0: continue
            for nbr in sim.neighbors[src]:
                if sim.belief[nbr] <= 0.5:
                    if spread_rng.random() > sim.share_prob[src]: continue
                    d = (sim.influence[src] * CRED * 1.5) / (1.0 + sim.skepticism[nbr])
                    if d > 0.03:
                        old = sim.belief[nbr]
                        sim.belief[nbr] = min(1.0, sim.belief[nbr] + d * sim.trust_score[nbr])
                        if old <= 0.5 and sim.belief[nbr] > 0.5:
                            nxt.append(nbr)
        active = list(set(nxt))
        history.append(int(np.sum(sim.belief > 0.5)))

    while len(history) < TIMESTEPS + 1:
        history.append(history[-1])
    return history

STRATEGIES = ['none', 'passive_degree', 'active_random', 'active_degree']
SLABELS = {
    'none':           'No Intervention',
    'passive_degree': 'Passive Degree (t=5)',
    'active_random':  'Active Random Cure',
    'active_degree':  'Active Degree Cure',
    'gnn_rl':         'GNN+RL (ours)',
}

baseline_results = defaultdict(lambda: defaultdict(list))
seed_nodes = {}

for gname, G in graphs.items():
    sn = sorted(G.degree(), key=lambda x: x[1], reverse=True)[SEED_NODE_RANK][0]
    seed_nodes[gname] = sn
    b_val = budget(G.number_of_nodes(), gname)
    print(f'\n{gname} | seed_node={sn} | budget={b_val}')
    for strategy in STRATEGIES:
        peaks = []
        for si in range(3):
            es = SEED_VALUE + si * 100
            peaks.append(max(run_baseline(G, sn, strategy, episode_seed=es, gname=gname)))
        baseline_results[gname][strategy] = peaks
        print(f'  {strategy:<22} peaks={peaks}  mean={np.mean(peaks):.0f}±{np.std(peaks):.0f}')

print('\nBaselines done.')


## Cell 5 — Models

- **RiskGNN**: 3-layer residual GCN (hidden=64) — fast yet expressive; ~8× fewer FLOPs than original 4-layer+GAT
- **DuelingDQN**: 128→64→32 with LayerNorm, 5 action heads
- **PERBuffer**: 20k capacity with β-annealing
- **5 actions**: wait | mute_small | mute_large | cure_infected | smart_burst_dense


In [ ]:
# ─── SumTree + PERBuffer ──────────────────────────────────────────────────────
class SumTree:
    """Priority tree with numpy-backed transition storage to minimise Python object overhead.
    Transitions stored as flat numpy arrays (s, a, r, ns, done) — avoids 15k Python tuples
    and tensor wrapper objects that cause heap fragmentation and high RSS."""
    S_DIM = 6  # state vector size

    def __init__(self, cap):
        self.cap  = cap
        self.tree = np.zeros(2 * cap - 1, dtype=np.float32)
        # Pre-allocate flat numpy arrays for each transition field
        self._s    = np.zeros((cap, self.S_DIM), dtype=np.float32)
        self._ns   = np.zeros((cap, self.S_DIM), dtype=np.float32)
        self._a    = np.zeros(cap, dtype=np.int64)
        self._r    = np.zeros(cap, dtype=np.float32)
        self._done = np.zeros(cap, dtype=np.float32)
        self.ptr  = 0; self.size = 0

    def _up(self, idx, delta):
        while idx > 0:
            idx = (idx - 1) // 2
            self.tree[idx] += delta

    def add(self, p, data):
        # data = (s_tensor_cpu, a_tensor, r_tensor, ns_tensor_cpu, done_tensor)
        s, a, r, ns, done = data
        self._s[self.ptr]    = s.numpy()
        self._ns[self.ptr]   = ns.numpy()
        self._a[self.ptr]    = int(a.item())
        self._r[self.ptr]    = float(r.item())
        self._done[self.ptr] = float(done.item())
        idx   = self.ptr + self.cap - 1
        delta = p - self.tree[idx]
        self.tree[idx] = p
        self._up(idx, delta)
        self.ptr  = (self.ptr + 1) % self.cap
        self.size = min(self.size + 1, self.cap)

    def get(self, s):
        idx = 0
        while idx < self.cap - 1:
            l = 2 * idx + 1
            if s <= self.tree[l]:
                idx = l
            else:
                s  -= self.tree[l]
                idx = l + 1
        data_idx = idx - (self.cap - 1)
        return data_idx, self.tree[idx], data_idx  # return index, not object

    @property
    def total(self): return self.tree[0]


class PERBuffer:
    def __init__(self, cap=PER_CAP, alpha=0.6, beta=0.4, beta_end=1.0, anneal_steps=50_000):
        self.tree     = SumTree(cap)
        self.alpha    = alpha
        self.beta     = beta
        self.beta_end = beta_end
        self.beta_inc = (beta_end - beta) / anneal_steps
        self.max_p    = 1.0

    def add(self, transition):
        self.tree.add(self.max_p ** self.alpha, transition)

    def sample(self, bs):
        if self.tree.size < bs: return None, None, None
        self.beta = min(self.beta_end, self.beta + self.beta_inc)
        idxs, iw = [], []
        seg = self.tree.total / bs
        for i in range(bs):
            s = random.uniform(seg * i, seg * (i + 1))
            idx, p, data_idx = self.tree.get(s)
            prob = p / (self.tree.total + 1e-8)
            iw.append((self.tree.size * prob + 1e-6) ** (-self.beta))
            idxs.append(data_idx)
        iw = np.array(iw, dtype=np.float32)
        iw /= (iw.max() + 1e-8)
        # Return batch as tensors built from numpy arrays (no Python object loop)
        ii = np.array(idxs)
        mb = (
            torch.from_numpy(self.tree._s[ii]),
            torch.from_numpy(self.tree._a[ii].reshape(-1,1)),
            torch.from_numpy(self.tree._r[ii].reshape(-1,1)),
            torch.from_numpy(self.tree._ns[ii]),
            torch.from_numpy(self.tree._done[ii].reshape(-1,1)),
        )
        return idxs, iw, mb

    def update(self, idxs, tds):
        for idx, td in zip(idxs, tds):
            p = (abs(td) + 1e-6) ** self.alpha
            self.max_p = max(self.max_p, p)
            leaf  = idx + self.tree.cap - 1
            delta = p - self.tree.tree[leaf]
            self.tree.tree[leaf] = p
            self.tree._up(leaf, delta)

    def __len__(self): return self.tree.size


# ─── Risk GNN (3-layer residual GCN, hidden=64) ───────────────────────────────
# FIX: Use LayerNorm instead of GroupNorm/BatchNorm.
#      GroupNorm(8, h) was mistakenly applied as bn(gcn_out.unsqueeze(0)),
#      which gives shape [1, N, h]. GroupNorm treats dim-1 as channels,
#      so it sees N nodes as C channels — fails when N % 8 != 0 (e.g. N=6299).
#      LayerNorm(h) normalises over the last dimension [N, h] — always correct,
#      works at any batch size and any graph size.
class RiskGNN(nn.Module):
    def __init__(self, in_dim=7 + SPEC_DIM, h=GNN_HIDDEN):
        super().__init__()
        self.proj = nn.Linear(in_dim, h, bias=False)
        self.c1   = GCNConv(in_dim, h)
        self.c2   = GCNConv(h, h)
        self.c3   = GCNConv(h, h // 2)
        self.c4   = GCNConv(h // 2, 1)
        # LayerNorm: normalises over feature dim — shape-agnostic, no num_groups issue
        self.bn1  = nn.LayerNorm(h)
        self.bn2  = nn.LayerNorm(h)
        self.bn3  = nn.LayerNorm(h // 2)
        self.drop = nn.Dropout(p=0.1)

    def forward(self, x, ei):
        res = self.proj(x)
        # LayerNorm operates on [N, h] directly — no unsqueeze/squeeze needed
        x1  = F.relu(self.bn1(self.c1(x, ei)))
        x2  = F.relu(self.bn2(self.c2(x1, ei))) + res
        x2  = self.drop(x2)
        x3  = F.relu(self.bn3(self.c3(x2, ei)))
        return torch.sigmoid(self.c4(x3, ei))


# ─── Dueling DDQN (5 actions, hidden=128) ────────────────────────────────────
N_ACTIONS = 5
# 0=wait  1=mute_small  2=mute_large  3=cure_infected  4=smart_burst_dense

class DuelingDQN(nn.Module):
    def __init__(self, s_dim=6, a_dim=N_ACTIONS, h=DQN_HIDDEN):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(s_dim, h),      nn.LayerNorm(h),      nn.ReLU(),
            nn.Linear(h, h // 2),    nn.LayerNorm(h // 2), nn.ReLU(),
            nn.Linear(h // 2, h // 4), nn.ReLU(),
        )
        self.V = nn.Linear(h // 4, 1)
        self.A = nn.Linear(h // 4, a_dim)

    def forward(self, x):
        h = self.shared(x)
        return self.V(h) + self.A(h) - self.A(h).mean(-1, keepdim=True)


_Q_MASK_FULL = torch.zeros(N_ACTIONS, dtype=torch.float)
_Q_MASK_NO_B = torch.tensor([0.] + [-1e9] * (N_ACTIONS - 1), dtype=torch.float)


def _topk_idx(arr, k):
    if k <= 0: return np.array([], dtype=np.int64)
    if k >= len(arr): return np.arange(len(arr))
    return np.argpartition(arr, -k)[-k:]


gnn_param_count = sum(p.numel() for p in RiskGNN().parameters())
dqn_param_count = sum(p.numel() for p in DuelingDQN().parameters())
print(f'RiskGNN params: {gnn_param_count:,}')
print(f'DuelingDQN params: {dqn_param_count:,}')
print('Models ready.')


## Cell 6 — Agent

In [ ]:
class MegaAgent:
    """
    MEGA Agent — GNN + Dueling DDQN for misinformation suppression.

    Key logic changes vs v3:
    ──────────────────────────────────────────────────────────────────────
    FIX-A  Mute actions now target INFECTED superspreaders only.
           Muting uninfected nodes zeros their influence before they ever
           spread, which has zero containment effect (they aren't spreading).
           Actions 1/2 now rank infected nodes by influence × degree.

    FIX-B  Episode budget density-scaled per graph.
           Sparse graphs (p2p, ca_grqc, avg_deg~6.6) get proportionally
           less budget than the dense facebook graph (avg_deg=43.7).
           Formula: ep_budget = b_step × TIMESTEPS × clamp(avg_deg/43.7, 0.15, 1.0) × 0.5
           This prevents the agent from curing 63% of all nodes on p2p,
           which was the root cause of the −8% suppression result.

    FIX-C  Reward adds an AUC penalty (sum of believers over episode).
           This punishes slow containment, not just the final peak.
           Encourages the agent to intervene EARLY and CONSISTENTLY.

    FIX-D  Reward terminal bonus: if epidemic extinct at episode end,
           award a strong positive signal (+2.0). Breaks the bimodal
           pattern by making "full containment" a clearly reachable goal.

    FIX-E  NUM_EVAL_SEEDS bumped to 5 via evaluate() default — more
           stable median at report time. Training unchanged.

    FIX-F  smart_burst (action 4) now also runs mute_superspreader,
           making it the strongest combined action.

    FIX-G  evaluate() accepts an optional `seed_node` kwarg so the caller
           can pass the same rank-3 seed node used by baselines.  Without
           this the agent evaluated from top_nodes[0] (rank-0 hub) while
           baselines started from rank-3 — a systematically harder scenario
           that inflated the agent's reported peak believers.

    FIX-H  Suppression % in Cell 7 is now computed against a matched
           `none` baseline run at the agent's own eval seeds (1000+s×100)
           rather than the baseline seeds (42+s×100).  This removes the
           seed-distribution confound on near-saturated graphs like Facebook.

    FIX-I  _vspread() accepts a `protected` node set.  Nodes that were
           cured or muted in the current timestep are excluded from the
           spread-susceptibility mask, giving interventions one full step
           of relief before they can be re-infected.  Critical on Facebook
           (avg_deg 43.7) where a single infected neighbour suffices to
           immediately reverse a cure.
    ──────────────────────────────────────────────────────────────────────
    """
    def __init__(self, G, graph_seed=0, gname=None, spec_emb=None,
                 pretrained_gnn=None, pretrained_dqn=None):
        self.G          = G
        self.graph_seed = graph_seed
        self.gname      = gname
        self.n          = len(G.nodes())
        self.spec_emb   = spec_emb if spec_emb is not None else np.zeros((self.n, SPEC_DIM), np.float32)

        feat_dim     = 7 + SPEC_DIM
        self.gnn     = RiskGNN(in_dim=feat_dim, h=GNN_HIDDEN).to(device)
        self.gnn_opt = optim.AdamW(self.gnn.parameters(), lr=5e-4, weight_decay=1e-4)
        self.gnn_sch = optim.lr_scheduler.CosineAnnealingLR(self.gnn_opt, T_max=400, eta_min=1e-5)

        self.net     = DuelingDQN(s_dim=6, a_dim=N_ACTIONS, h=DQN_HIDDEN).to(device)
        self.opt     = optim.AdamW(self.net.parameters(), lr=3e-4, weight_decay=1e-4)
        self.sch     = optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=400, eta_min=1e-5)

        self.tnet    = copy.deepcopy(self.net).to(device); self.tnet.eval()
        self.buf     = PERBuffer(cap=PER_CAP)
        self.sim     = Sim(G, graph_seed=graph_seed)
        self._curriculum = pretrained_gnn is not None

        if pretrained_gnn is not None:
            try:
                self.gnn.load_state_dict(pretrained_gnn, strict=False)
                print('  GNN weights transferred (curriculum).')
            except Exception as e:
                print(f'  GNN transfer skipped: {e}')
        if pretrained_dqn is not None:
            try:
                self.net.load_state_dict(pretrained_dqn, strict=False)
                self.tnet.load_state_dict(self.net.state_dict())
                print('  DQN weights transferred (curriculum).')
            except Exception as e:
                print(f'  DQN transfer skipped: {e}')

        ei_list = [[a, b] for a, b in G.edges()] + [[b, a] for a, b in G.edges()]
        self.ei = torch.tensor(ei_list, dtype=torch.long).t().contiguous().to(device)

        src_arr = np.array([s for s, _ in G.edges()], dtype=np.int32)
        dst_arr = np.array([d for _, d in G.edges()], dtype=np.int32)
        self._src_all = np.concatenate([src_arr, dst_arr])
        self._dst_all = np.concatenate([dst_arr, src_arr])

        cpath = f'cc_{self.n}_{graph_seed}.npz'
        if os.path.exists(cpath):
            c = np.load(cpath, allow_pickle=True)
            self.deg_n = c['deg']; self.bw = c['bw']
        else:
            dg = dict(G.degree()); mx = max(dg.values()) or 1
            self.deg_n = np.array([dg[v] / mx for v in range(self.n)], dtype=np.float32)
            if self.n <= 6000:
                t0 = time.time()
                bwd = nx.betweenness_centrality(G, normalized=True,
                                                k=min(200, self.n), seed=graph_seed)
                self.bw = np.array([bwd[v] for v in range(self.n)], dtype=np.float32)
                print(f'  betweenness done in {time.time()-t0:.1f}s')
                del bwd
            else:
                self.bw = self.deg_n.copy()
                print(f'  Facebook: using degree proxy for betweenness (n={self.n})')
            np.savez(cpath, deg=self.deg_n, bw=self.bw)

        self.mat = self._build_mat()
        self._mat_dirty = False

        self.top_nodes  = [n for n, _ in sorted(G.degree(), key=lambda x: x[1], reverse=True)[:15]]
        self._feat_dim  = feat_dim
        self._feat_buf  = np.empty((self.n, feat_dim), dtype=np.float32)
        self._avg_deg   = 2 * G.number_of_edges() / self.n
        self._b_step    = budget(self.n, self.gname)

        # FIX-B: density-scaled episode budget
        # Sparse graphs (avg_deg~6) get ~15% of facebook's budget multiplier.
        # Prevents the agent curing 60%+ of all nodes on sparse topologies.
        _FACEBOOK_AVG_DEG = 43.7
        _density_factor   = min(1.0, max(0.30, self._avg_deg / _FACEBOOK_AVG_DEG))  # raised floor: p2p gets ~30% of FB budget, enough for meaningful intervention
        self._ep_budget   = self._b_step * TIMESTEPS * _density_factor * 0.5
        print(f'  ep_budget={self._ep_budget:.0f}  (density_factor={_density_factor:.2f})')

        self._cached_risks   = np.zeros(self.n, dtype=np.float32)
        self._risk_cache_age = RISK_CACHE_STEPS
        self._reward_log = []
        self._loss_log   = []

    # ── Internal helpers ──────────────────────────────────────────────────────
    def _build_mat(self):
        s, d = self._src_all, self._dst_all
        w    = self.sim.influence[s] * CRED * 1.5 * self.sim.share_prob[s]
        val  = w / (1.0 + self.sim.skepticism[d]) * self.sim.trust_score[d]
        mask = val > 0.03
        return sp.csr_matrix((val[mask], (d[mask], s[mask])),
                             shape=(self.n, self.n), dtype=np.float32)

    def _feats(self, frontier):
        b = self._feat_buf
        b[:, 0] = (self.sim.belief > 0.5).astype(np.float32)
        b[:, 1] = 0.0
        if frontier: b[frontier, 1] = 1.0
        b[:, 2] = self.deg_n
        b[:, 3] = self.bw
        b[:, 4] = self.sim.belief
        b[:, 5] = self.sim.skepticism
        b[:, 6] = self.sim.influence
        b[:, 7:7 + SPEC_DIM] = self.spec_emb
        return torch.from_numpy(b.copy()).to(device)

    def _get_risks(self, frontier, force=False):
        self._risk_cache_age += 1
        if force or self._risk_cache_age >= RISK_CACHE_STEPS:
            X = self._feats(frontier)
            self.gnn.eval()
            with torch.no_grad():
                self._cached_risks = self.gnn(X, self.ei).cpu().numpy().flatten()
            self._risk_cache_age = 0
        return self._cached_risks

    def _build_state(self, believers, rem_budget, frontier, risks=None):
        if risks is None: risks = self._cached_risks
        mask  = self.sim.belief <= 0.5
        avg_r = float(risks[mask].mean()) if mask.any() else 0.0
        max_r = float(risks[mask].max())  if mask.any() else 0.0
        return torch.tensor([
            avg_r,
            max_r,
            believers / self.n,
            rem_budget / max(self._ep_budget, 1.0),
            len(frontier) / self.n,
            min(1.0, self._avg_deg / 50.0),
        ], dtype=torch.float32, device=device)

    # ── Actions ───────────────────────────────────────────────────────────────
    def _mute_superspreaders(self, risks, n_targets):
        """FIX-A: Silence the most dangerous INFECTED nodes (influence × degree).
        These are the nodes actively amplifying the cascade right now.
        Muting uninfected nodes is useless — they aren't spreading.
        Returns (cost, acted_indices) so the caller can pass acted_indices
        to _vspread as `protected` (FIX-I).
        """
        infected = np.where(self.sim.belief > 0.5)[0]
        if len(infected) == 0: return 0, np.array([], dtype=np.int64)
        # Score = GNN risk × influence × degree — captures both current danger
        # and structural amplification potential
        score = (risks[infected]
                 * self.sim.influence[infected]
                 * self.deg_n[infected])
        k   = min(len(infected), max(1, int(n_targets)))
        idx = infected[_topk_idx(score, k)]
        acted = []
        for v in idx:
            if self.sim.influence[v] > 0.0:
                self.sim.influence[v] = 0.0; acted.append(v)
        if acted: self._mat_dirty = True
        return len(acted), np.array(acted, dtype=np.int64)

    def _apply_cure(self, risks, rem_budget, b_step):
        """Cure + dampen influence of top-risk infected nodes.
        Returns (cost, acted_indices) so the caller can pass acted_indices
        to _vspread as `protected` (FIX-I).
        """
        act_mask = (self.sim.belief > 0.5) & (risks > 0.1)  # lower threshold: act on more infected nodes
        eligible = np.where(act_mask)[0]
        if len(eligible) == 0:
            eligible = np.where(self.sim.belief > 0.5)[0]
        if len(eligible) == 0: return 0, np.array([], dtype=np.int64)
        amt = min(len(eligible), int(rem_budget), b_step)
        idx = eligible[_topk_idx(risks[eligible], amt)]
        self.sim.belief[idx]    = np.maximum(0.0, self.sim.belief[idx] - 0.85)
        self.sim.influence[idx] = self.sim.influence[idx] * 0.3
        if len(idx) > 0: self._mat_dirty = True
        return len(idx), idx

    def _step_action(self, a, risks, rem_budget):
        """Execute action.  Returns (cost, action_penalty, protected_nodes).

        FIX-I: protected_nodes is the union of all node indices directly
        acted on this step.  Pass it to _vspread to prevent same-timestep
        re-infection on dense graphs.
        """
        b_step = self._b_step
        if rem_budget <= 0: return 0, 0.0, np.array([], dtype=np.int64)
        cost = 0
        protected = np.array([], dtype=np.int64)
        if   a == 1:
            # Mute small: silence top-b_step superspreaders
            cost, protected = self._mute_superspreaders(risks, min(b_step, int(rem_budget)))
        elif a == 2:
            # Mute large: silence top 3×b_step superspreaders (heavy intervention)
            cost, protected = self._mute_superspreaders(risks, min(b_step * 3, int(rem_budget)))
        elif a == 3:
            # Cure: reduce belief of top-risk infected nodes
            cost, protected = self._apply_cure(risks, rem_budget, b_step)
        elif a == 4:
            # Smart burst (FIX-F): cure top half + mute superspreaders bottom half
            half           = max(1, b_step // 2)
            cost_c, prot_c = self._apply_cure(risks, min(rem_budget, half), half)
            cost_m, prot_m = self._mute_superspreaders(risks, min(half, int(max(0, rem_budget - cost_c))))
            cost      = cost_c + cost_m
            protected = np.union1d(prot_c, prot_m)
        action_cost = {0: 0.0, 1: 1.5, 2: 3.5, 3: 1.0, 4: 2.5}.get(a, 0.0)
        return cost, action_cost, protected

    def _vspread(self, frontier, protected=None):
        """Vectorised spread step.

        FIX-I: `protected` is an optional array of node indices that were
        acted on (cured or muted) this timestep.  On dense graphs like
        Facebook a cured node has ~44 infected neighbours, each contributing
        ~0.5 belief, so without protection it is immediately re-infected in
        the same call — making every cure action pointless.  Excluding
        protected nodes from the `old_inf` mask gives the intervention a
        full timestep of relief before they can be re-infected.
        """
        if not frontier: return []
        v = np.zeros(self.n, dtype=np.float32)
        v[frontier] = 1.0
        v[self.sim.belief <= 0.5]    = 0.0
        v[self.sim.influence == 0.0] = 0.0
        if self._mat_dirty:
            self.mat = self._build_mat()
            self._mat_dirty = False
        delta   = self.mat.dot(v)
        old_inf = self.sim.belief <= 0.5
        if protected is not None and len(protected) > 0:
            old_inf[protected] = False   # FIX-I: one-step immunity for acted-on nodes
        self.sim.belief = np.minimum(1.0, self.sim.belief + delta * old_inf)
        return list(np.where(self.sim.belief > 0.5)[0])

    # ── Training ──────────────────────────────────────────────────────────────
    def train(self, episodes=1000, bs=64, gamma=0.99, eps=1.0, decay=0.997,
              log_every=100, ckpt_every=500, gnn_update_freq=4):
        if self._curriculum and eps == 1.0:
            eps = 0.4
            print(f'  Curriculum warm-start: eps={eps}')
        print(f'  training {episodes} eps | ep_budget={self._ep_budget:.0f} | gnn_cache={RISK_CACHE_STEPS}')
        q_full = _Q_MASK_FULL.to(device)
        q_no_b = _Q_MASK_NO_B.to(device)
        t_start = time.time()

        for ep in range(episodes):
            top      = self.top_nodes[ep % len(self.top_nodes)]
            frontier = self.sim.reset(top, episode_seed=ep, pre_spread=PRE_SPREAD_STEPS)
            self.mat = self._build_mat(); self._mat_dirty = False

            rem_budget   = self._ep_budget
            believers    = int(np.sum(self.sim.belief > 0.5))
            prev_b       = believers
            peak_this_ep = max(believers, 1)
            ep_beliefs   = []
            ep_reward    = 0.0
            auc_sum      = 0.0   # FIX-C: track AUC for reward

            risks = self._get_risks(frontier, force=True)
            s     = self._build_state(believers, rem_budget, frontier, risks)

            for t in range(TIMESTEPS):
                if not frontier: break

                if random.random() < eps:
                    a = 0 if rem_budget <= 0 else random.randint(0, N_ACTIONS - 1)
                else:
                    self.net.eval()
                    with torch.no_grad():
                        mask = q_no_b if rem_budget <= 0 else q_full
                        a    = int(torch.argmax(self.net(s.unsqueeze(0)).squeeze(0) + mask).item())

                cost, action_penalty, protected = self._step_action(a, risks, rem_budget)
                rem_budget = max(0.0, rem_budget - cost)
                ep_beliefs.append(self.sim.belief.copy())

                frontier  = self._vspread(frontier, protected=protected)  # FIX-I
                believers = int(np.sum(self.sim.belief > 0.5))
                peak_this_ep = max(peak_this_ep, believers)
                auc_sum  += believers / self.n   # FIX-C

                risks = self._get_risks(frontier)
                ns    = self._build_state(believers, rem_budget, frontier, risks)

                # FIX-C/D: reward = suppression delta + AUC penalty + terminal bonus
                delta_b   = prev_b - believers
                prev_b    = believers
                load_frac = believers / peak_this_ep if peak_this_ep > 0 else 0.0
                _density_scale = max(1.0, np.log(self._avg_deg / 6.0 + 1.0))  # ~2.0 for FB, ~1.0 sparse
                r = (
                    - load_frac * 0.8 * _density_scale          # penalise current load (density-scaled)
                    + (delta_b / max(peak_this_ep, 1)) * 5.0    # STRONGER suppression reward — key signal
                    - (auc_sum / max(TIMESTEPS, 1)) * 0.10       # softer AUC penalty: don't over-penalise early load
                    - action_penalty * 0.005                     # halved action cost: encourage intervention
                )
                # FIX-D: terminal extinction bonus
                done_flag = 0.0 if frontier else 1.0
                if done_flag == 1.0:
                    r += 3.0  # stronger extinction signal
                ep_reward += r

                self.buf.add((s.cpu(), torch.tensor([a]), torch.tensor([r]),
                              ns.cpu(), torch.tensor([done_flag])))
                s = ns

            self._reward_log.append(ep_reward)

            # DQN update
            idxs, bIW, mb = self.buf.sample(bs)
            if mb is not None:
                bIW = torch.tensor(bIW, device=device)
                bS, bA, bR, bNS, bD = [x.to(device) for x in mb]
                self.net.train()
                cq = self.net(bS).gather(1, bA)
                with torch.no_grad():
                    best_a = self.net(bNS).argmax(1, keepdim=True)
                    tq     = bR + gamma * self.tnet(bNS).gather(1, best_a) * (1 - bD)
                td   = (tq - cq).detach().cpu().numpy().flatten()
                self.buf.update(idxs, td)
                loss = (bIW * F.mse_loss(cq, tq, reduction='none').squeeze()).mean()
                self.opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), 1.0)
                self.opt.step()
                self._loss_log.append(loss.item())

            # GNN supervised update — lookahead=10, class-balanced BCE
            if ep % gnn_update_freq == 0 and len(ep_beliefs) >= 2:
                lookahead  = 5 if self.gname == 'facebook' else (15 if self.gname == 'p2p_gnutella' else 10)
                n_samples  = min(8 if self.gname == 'facebook' else 6, len(ep_beliefs))
                sample_idx = random.sample(range(len(ep_beliefs)), n_samples)
                gnn_loss   = torch.tensor(0.0, device=device)
                self.gnn.train()
                for si in sample_idx:
                    t_idx = min(si + lookahead, len(ep_beliefs) - 1)
                    buf   = self._feat_buf.copy()
                    buf[:, 0] = (ep_beliefs[si] > 0.5).astype(np.float32)
                    buf[:, 1] = 0.0
                    buf[:, 2] = self.deg_n; buf[:, 3] = self.bw
                    buf[:, 4] = ep_beliefs[si]; buf[:, 5] = self.sim.skepticism
                    buf[:, 6] = self.sim.influence
                    buf[:, 7:7 + SPEC_DIM] = self.spec_emb
                    x_in   = torch.from_numpy(buf).to(device)
                    labels = torch.from_numpy((ep_beliefs[t_idx] > 0.5).astype(np.float32)).to(device)
                    n_pos  = labels.sum().item()
                    n_neg  = self.n - n_pos
                    pos_w  = torch.tensor(n_neg / max(n_pos, 1), device=device)
                    preds = self.gnn(x_in, self.ei).squeeze()
                    bce   = F.binary_cross_entropy(preds, labels, reduction='none')
                    w     = torch.where(labels == 1, pos_w, torch.ones_like(pos_w))
                    gnn_loss = gnn_loss + (bce * w).mean()
                self.gnn_opt.zero_grad()
                (gnn_loss / n_samples).backward()
                nn.utils.clip_grad_norm_(self.gnn.parameters(), 1.0)
                self.gnn_opt.step()

            if (ep + 1) % 50 == 0:
                self.tnet.load_state_dict(self.net.state_dict())
                self.sch.step(); self.gnn_sch.step()

            eps = max(0.05, eps * decay)

            if (ep + 1) % log_every == 0:
                elapsed = time.time() - t_start
                avg_r   = np.mean(self._reward_log[-log_every:]) if self._reward_log else 0
                avg_l   = np.mean(self._loss_log[-log_every:])   if self._loss_log   else 0
                eta_min = (elapsed / (ep + 1)) * (episodes - ep - 1) / 60
                print(f'    ep {ep+1:>4}/{episodes}  eps={eps:.3f}  '
                      f'avg_r={avg_r:.3f}  loss={avg_l:.4f}  '
                      f'elapsed={elapsed/60:.1f}min  ETA={eta_min:.1f}min')

            if (ep + 1) % ckpt_every == 0:
                self.save_ckpt(f'weights/{self.gname}_ep{ep+1}')
                print(f'    ✓ checkpoint @ ep {ep+1}')

        print(f'  training complete in {(time.time()-t_start)/60:.1f} min')

    # ── Evaluation ────────────────────────────────────────────────────────────
    def evaluate(self, eval_seed=0, seed_node=None):
        """Run one greedy evaluation episode.

        FIX-G: `seed_node` lets the caller pass the same rank-3 seed node
        used by baselines, ensuring the comparison is on identical epidemic
        starts.  Falls back to top_nodes[0] when not provided (training
        curriculum behaviour is unchanged).
        """
        self.sim = Sim(self.G, graph_seed=self.graph_seed)
        top      = seed_node if seed_node is not None else self.top_nodes[0]  # FIX-G
        frontier = self.sim.reset(top, episode_seed=eval_seed, pre_spread=PRE_SPREAD_STEPS)
        self.mat = self._build_mat(); self._mat_dirty = False
        # FIX-E: correct initial believer count after pre_spread
        believers = int(np.sum(self.sim.belief > 0.5))
        log       = [believers]
        rem_b     = self._ep_budget
        q_no_b    = _Q_MASK_NO_B.to(device)
        q_full    = _Q_MASK_FULL.to(device)

        risks = self._get_risks(frontier, force=True)
        for t in range(TIMESTEPS):
            if not frontier: break
            s = self._build_state(believers, rem_b, frontier, risks)
            self.net.eval()
            with torch.no_grad():
                mask = q_no_b if rem_b <= 0 else q_full
                a    = int(torch.argmax(self.net(s.unsqueeze(0)).squeeze(0) + mask).item())
            cost, _, protected = self._step_action(a, risks, rem_b)
            rem_b   = max(0.0, rem_b - cost)
            frontier  = self._vspread(frontier, protected=protected)  # FIX-I
            believers = int(np.sum(self.sim.belief > 0.5))
            risks = self._get_risks(frontier)
            log.append(believers)

        while len(log) < TIMESTEPS + 1:
            log.append(log[-1])
        return log

    def save_ckpt(self, prefix):
        torch.save(self.gnn.state_dict(), prefix + '_gnn.pt')
        torch.save(self.net.state_dict(), prefix + '_dqn.pt')

    def load_ckpt(self, prefix):
        self.gnn.load_state_dict(torch.load(prefix + '_gnn.pt', map_location=device))
        self.net.load_state_dict(torch.load(prefix + '_dqn.pt', map_location=device))
        self.tnet.load_state_dict(self.net.state_dict())
        print(f'  loaded: {prefix}')


print('MegaAgent ready.')


## Cell 7 — Training + Ensemble Evaluation

**Curriculum**: p2p_gnutella → ca_grqc → facebook (best ca_grqc weights transferred).  
**Ensemble**: 3 train seeds × 7 eval seeds = 21 evaluations per graph.  
**Expected total wall-clock**: ~3.5–5 h on Kaggle T4.


In [ ]:
rl_results  = defaultdict(list)
all_agents  = {}
summary     = {}

pretrained_gnn_state = None
pretrained_dqn_state = None
t_global_start       = time.time()

for gname in GRAPH_ORDER:
    G        = graphs[gname]
    spec_emb = spectral[gname]
    episodes = EPISODE_SCHEDULE[gname]
    guf      = GNN_UPDATE_FREQ[gname]
    b_val    = budget(G.number_of_nodes(), gname)
    n_nodes  = G.number_of_nodes()

    print(f'\n{"="*62}')
    print(f'Graph: {gname}  ({n_nodes} nodes, {G.number_of_edges()} edges)')
    print(f'  budget/step : {b_val}')
    print(f'  episodes    : {episodes} per seed  ({NUM_TRAIN_SEEDS} seeds)')
    print(f'  curriculum  : {"YES — pretrained weights" if pretrained_gnn_state else "cold start"}')
    print(f'{"="*62}')

    agents_for_graph = []

    for seed_i in range(NUM_TRAIN_SEEDS):
        ep_seed = SEED_VALUE + seed_i * 1000
        print(f'\n  [seed {seed_i+1}/{NUM_TRAIN_SEEDS}  ep_seed={ep_seed}]')
        torch.manual_seed(ep_seed); np.random.seed(ep_seed); random.seed(ep_seed)

        agent = MegaAgent(
            G, graph_seed=seed_i, gname=gname, spec_emb=spec_emb,
            pretrained_gnn=pretrained_gnn_state if gname in ('facebook', 'ca_grqc') else None,
            pretrained_dqn=pretrained_dqn_state if gname in ('facebook', 'ca_grqc') else None,
        )
        agent.train(
            episodes        = episodes,
            bs              = 64,
            gamma           = 0.99,
            eps             = 1.0,
            decay           = 0.9975 if gname == 'facebook' else (0.9930 if gname == 'p2p_gnutella' else 0.993),
            log_every       = 100,
            ckpt_every      = 500,
            gnn_update_freq = guf,
        )
        agents_for_graph.append(agent)

    all_agents[gname] = agents_for_graph

    # ── Ensemble evaluation ──────────────────────────────────────────────────
    print(f'\n  Evaluating ensemble ({NUM_TRAIN_SEEDS} agents × {NUM_EVAL_SEEDS} seeds)...')
    # FIX-G: evaluate on the same seed node as baselines (rank-3 = seed_nodes[gname])
    eval_peaks = []
    for agent in agents_for_graph:
        for eval_s in range(NUM_EVAL_SEEDS):
            log = agent.evaluate(eval_seed=1000 + eval_s * 100,
                                 seed_node=seed_nodes[gname])   # FIX-G
            eval_peaks.append(max(log))
            rl_results[gname].append(log)

    med = float(np.median(eval_peaks))
    std = float(np.std(eval_peaks))

    # FIX-H: compute suppression against a 'none' baseline at the SAME eval seeds
    # so the comparison is on identical epidemic trajectories (same seed node,
    # same episode seeds).  The original baseline_results used seeds 42/142/242;
    # agent evals use 1000/1100/…  On near-saturated graphs the seed distribution
    # shifts peak believers enough to swamp a small suppression signal.
    matched_none_peaks = []
    for eval_s in range(NUM_EVAL_SEEDS):
        es  = 1000 + eval_s * 100
        h   = run_baseline(G, seed_nodes[gname], 'none',
                           episode_seed=es, gname=gname)       # FIX-H
        matched_none_peaks.append(max(h))
    none_med = float(np.median(matched_none_peaks))            # FIX-H
    supp     = (none_med - med) / none_med * 100 if none_med > 0 else 0.0

    summary[gname] = {
        'gnn_rl': {'median': med, 'std': std, 'supp': supp, 'peaks': eval_peaks},
        'none':   {'median': none_med},
    }
    print(f'  ✅ GNN+RL: median={med:.0f}  std={std:.0f}  suppression={supp:.1f}%')

    for i, agent in enumerate(agents_for_graph):
        agent.save_ckpt(f'weights/{gname}_final_seed{i}')

    # Curriculum: after ca_grqc, pass best agent weights to Facebook
    if gname in ('p2p_gnutella', 'ca_grqc'):
        seed_medians = []
        for i, agent in enumerate(agents_for_graph):
            ps = [max(rl_results[gname][i * NUM_EVAL_SEEDS + j]) for j in range(NUM_EVAL_SEEDS)]
            seed_medians.append(float(np.median(ps)))
        best_idx = int(np.argmin(seed_medians))
        pretrained_gnn_state = copy.deepcopy(agents_for_graph[best_idx].gnn.state_dict())
        pretrained_dqn_state = copy.deepcopy(agents_for_graph[best_idx].net.state_dict())
        next_graph = 'ca_grqc' if gname == 'p2p_gnutella' else 'facebook'
        print(f'  Curriculum: seed {best_idx} weights passed to {next_graph} agent.')

    elapsed = (time.time() - t_global_start) / 60
    print(f'  Total elapsed: {elapsed:.1f} min')

    # Free GPU memory and RAM between graphs
    if gname != 'facebook':
        for agent in agents_for_graph:
            agent.gnn.cpu(); agent.net.cpu(); agent.tnet.cpu()
            # Free the replay buffer (largest RAM user per agent)
            del agent.buf; agent.buf = None
            # Compact logs to numpy arrays so they stay accessible for plots
            # but release the Python list memory
            agent._reward_log = np.array(agent._reward_log, dtype=np.float32)
            agent._loss_log   = np.array(agent._loss_log,   dtype=np.float32)
        torch.cuda.empty_cache()
        gc.collect()
    else:
        # After facebook training, free buffers too (only weights needed for eval/plots)
        for agent in agents_for_graph:
            del agent.buf; agent.buf = None
            agent._reward_log = np.array(agent._reward_log, dtype=np.float32)
            agent._loss_log   = np.array(agent._loss_log,   dtype=np.float32)
        torch.cuda.empty_cache()
        gc.collect()

print(f'\n✅ All training complete. Total: {(time.time()-t_global_start)/60:.1f} min')


## Cell 8 — Visualisations

In [ ]:
COLORS = {
    'none':           '#e74c3c',
    'passive_degree': '#e67e22',
    'active_random':  '#f1c40f',
    'active_degree':  '#9b59b6',
    'gnn_rl':         '#2ecc71',
}

# ── Plot 1: Suppression curves ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, gname in zip(axes, GRAPH_ORDER):
    G = graphs[gname]
    for strat in STRATEGIES:
        curves = [run_baseline(G, seed_nodes[gname], strat,
                               episode_seed=SEED_VALUE + si * 100, gname=gname)
                  for si in range(3)]
        med = np.median(curves, axis=0)
        ax.plot(med, color=COLORS[strat], lw=1.5, ls='--',
                label=SLABELS[strat], alpha=0.75)
    rl_curves = rl_results[gname]
    if rl_curves:
        rl_med = np.median(rl_curves, axis=0)
        rl_lo  = np.percentile(rl_curves, 10, axis=0)
        rl_hi  = np.percentile(rl_curves, 90, axis=0)
        ts = np.arange(len(rl_med))
        ax.plot(ts, rl_med, color=COLORS['gnn_rl'], lw=2.5, label='GNN+RL (ours)')
        ax.fill_between(ts, rl_lo, rl_hi, color=COLORS['gnn_rl'], alpha=0.2, label='10–90 pct')
    supp = summary[gname]['gnn_rl']['supp']
    ax.set_title(f"{gname.replace('_',' ').title()}\nSuppression: {supp:.1f}%",
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Timestep'); ax.set_ylabel('Believers')
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.suptitle('Misinformation Suppression Curves — GNN+RL vs Baselines', fontsize=14)
plt.tight_layout(); plt.savefig('plots/curves.png', dpi=150); plt.show()

# ── Plot 2: Bar chart ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
all_strats = STRATEGIES + ['gnn_rl']
x = np.arange(len(GRAPH_ORDER)); w = 0.14
for i, strat in enumerate(all_strats):
    vals = []
    for gname in GRAPH_ORDER:
        if strat == 'gnn_rl':
            vals.append(summary[gname]['gnn_rl']['median'])
        else:
            vals.append(np.median(baseline_results[gname][strat]))
    ax.bar(x + i * w, vals, w * 0.9, label=SLABELS[strat],
           color=COLORS[strat], edgecolor='black', lw=0.5)
ax.set_xticks(x + w * 2)
ax.set_xticklabels([g.replace('_', '\n') for g in GRAPH_ORDER])
ax.set_ylabel('Median Peak Believers')
ax.set_title('Peak Believer Comparison — All Strategies')
ax.legend(loc='upper right', fontsize=8); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('plots/bar.png', dpi=150); plt.show()

# ── Plot 3: Suppression heatmap ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
row_labels = STRATEGIES + ['gnn_rl']
supp_mat   = []
for strat in row_labels:
    row = []
    for gname in GRAPH_ORDER:
        none_m = np.median(baseline_results[gname]['none'])
        peak   = (summary[gname]['gnn_rl']['median'] if strat == 'gnn_rl'
                  else np.median(baseline_results[gname][strat]))
        row.append((none_m - peak) / none_m * 100 if none_m > 0 else 0.0)
    supp_mat.append(row)
supp_mat = np.array(supp_mat)
im = ax.imshow(supp_mat, cmap='RdYlGn', vmin=-20, vmax=100)
ax.set_xticks(range(3)); ax.set_xticklabels([g.replace('_', '\n') for g in GRAPH_ORDER])
ax.set_yticks(range(len(row_labels))); ax.set_yticklabels([SLABELS[s] for s in row_labels])
for i in range(len(row_labels)):
    for j in range(3):
        ax.text(j, i, f'{supp_mat[i,j]:.1f}%', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax, label='Suppression %')
ax.set_title('Suppression % Heatmap — Strategy × Graph')
plt.tight_layout(); plt.savefig('plots/heatmap.png', dpi=150); plt.show()

# ── Plot 4: Training reward curves ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, gname in zip(axes, GRAPH_ORDER):
    for i, agent in enumerate(all_agents[gname]):
        if len(agent._reward_log):
            smooth = np.convolve(agent._reward_log, np.ones(30)/30, mode='valid')
            ax.plot(smooth, label=f'seed {i}', alpha=0.8)
    ax.set_title(f"{gname.replace('_',' ').title()}\nTraining Reward (smoothed w=30)")
    ax.set_xlabel('Episode'); ax.set_ylabel('Avg Reward')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle('Training Reward Curves', fontsize=13)
plt.tight_layout(); plt.savefig('plots/reward_curves.png', dpi=150); plt.show()

# ── Plot 5: DQN loss curves ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, gname in zip(axes, GRAPH_ORDER):
    for i, agent in enumerate(all_agents[gname]):
        if len(agent._loss_log):
            smooth = np.convolve(agent._loss_log, np.ones(50)/50, mode='valid')
            ax.plot(smooth, label=f'seed {i}', alpha=0.8)
    ax.set_title(f"{gname.replace('_',' ').title()}\nDQN TD Loss")
    ax.set_xlabel('Update step'); ax.set_ylabel('MSE Loss')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle('DQN Training Loss', fontsize=13)
plt.tight_layout(); plt.savefig('plots/loss_curves.png', dpi=150); plt.show()

# ── Plot 6: Budget usage (Facebook) ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
gname  = 'facebook'
agent  = all_agents[gname][0]
G      = graphs[gname]
sim    = Sim(G, graph_seed=0)
top    = seed_nodes[gname]              # FIX-G: same seed node as baselines
frontier = sim.reset(top, episode_seed=0)
agent.sim = sim; agent.mat = agent._build_mat()
max_b  = agent._ep_budget  # use the same ep_budget as training
rem_b  = float(max_b); believers = int(np.sum(agent.sim.belief > 0.5))
budget_log = [rem_b]; action_log = []
q_no_b = _Q_MASK_NO_B.to(device); q_full = _Q_MASK_FULL.to(device)
risks  = agent._get_risks(frontier, force=True)
for t in range(TIMESTEPS):
    s = agent._build_state(believers, rem_b, frontier, risks)
    agent.net.eval()
    with torch.no_grad():
        mask = q_no_b if rem_b <= 0 else q_full
        a    = int(torch.argmax(agent.net(s.unsqueeze(0)).squeeze(0) + mask).item())
    action_log.append(a)
    cost, _, prot = agent._step_action(a, risks, rem_b)
    rem_b   = max(0.0, rem_b - cost)
    frontier  = agent._vspread(frontier, protected=prot)  # FIX-I
    believers = int(np.sum(agent.sim.belief > 0.5))
    risks = agent._get_risks(frontier)
    budget_log.append(rem_b)

ACTION_COLORS = ['#95a5a6', '#3498db', '#2980b9', '#e74c3c', '#e67e22']
ACTION_NAMES  = ['Wait', 'Mute Small', 'Mute Large', 'Cure', 'Smart Burst']
ax.plot(budget_log, color='#2c3e50', lw=2, label='Remaining Budget')
for t, a in enumerate(action_log):
    ax.axvline(t, color=ACTION_COLORS[a], alpha=0.3, lw=4)
patches = [mpatches.Patch(color=ACTION_COLORS[i], label=ACTION_NAMES[i]) for i in range(N_ACTIONS)]
patches.append(mpatches.Patch(color='#2c3e50', label='Budget'))
ax.legend(handles=patches, fontsize=9, loc='lower left')
ax.set_title('Facebook — Budget & Action Timeline (best agent)')
ax.set_xlabel('Timestep'); ax.set_ylabel('Remaining Budget'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('plots/budget_usage.png', dpi=150); plt.show()

print('\n✅ All 6 plots saved to plots/')


## Cell 9 — Summary + Export

In [ ]:
# ── Results table ─────────────────────────────────────────────────────────────
SEP = '=' * 82
print(SEP)
print(f"{'Graph':<14} {'Strategy':<24} {'Median Peak':>11} {'Std':>7} {'Suppression':>12}")
print(SEP)

for gname in GRAPH_ORDER:
    G = graphs[gname]
    print(f"{gname} ({G.number_of_nodes()} nodes)")
    none_m = float(np.median(baseline_results[gname]['none']))
    for strat in STRATEGIES:
        peak = float(np.median(baseline_results[gname][strat]))
        std_ = float(np.std(baseline_results[gname][strat]))
        supp = (none_m - peak) / none_m * 100 if none_m > 0 else 0.0
        print(f"  {SLABELS[strat]:<22} {peak:>11.0f} {std_:>7.0f} {supp:>11.1f}%")
    rl = summary[gname]['gnn_rl']
    print(f"  {'GNN+RL (ours)':<22} {rl['median']:>11.0f} {rl['std']:>7.0f} {rl['supp']:>11.1f}%")
    print('-' * 82)

print(SEP)
print('\nKEY OPTIMISATIONS (this run vs original MEGA_V1):')
print(f'  GNN risk caching   : every {RISK_CACHE_STEPS} steps (vs every step) — ~{RISK_CACHE_STEPS}x fewer GNN calls')
print(f'  GNN hidden dim     : {GNN_HIDDEN} (vs 128) — ~4x fewer params')
print(f'  DQN hidden dim     : {DQN_HIDDEN} (vs 256)')
print(f'  Episodes (facebook): {EPISODE_SCHEDULE["facebook"]} (vs 3000) + curriculum transfer')
print(f'  Train seeds        : {NUM_TRAIN_SEEDS} (vs 3)')
print(f'  Eval seeds         : {NUM_EVAL_SEEDS} (vs 5)')

# ── Save JSON ─────────────────────────────────────────────────────────────────
json_summary = {}
for gname in GRAPH_ORDER:
    json_summary[gname] = {
        'nodes':           graphs[gname].number_of_nodes(),
        'edges':           graphs[gname].number_of_edges(),
        'budget_per_step': budget(graphs[gname].number_of_nodes(), gname),
        'episodes':        EPISODE_SCHEDULE[gname],
        'gnn_rl': {
            'median':          summary[gname]['gnn_rl']['median'],
            'std':             summary[gname]['gnn_rl']['std'],
            'suppression_pct': summary[gname]['gnn_rl']['supp'],
        },
        'baselines': {s: float(np.median(baseline_results[gname][s])) for s in STRATEGIES},
    }
with open('summary_results.json', 'w') as f:
    json.dump(json_summary, f, indent=2)
print('\n✓ summary_results.json saved.')

# ── Archive ───────────────────────────────────────────────────────────────────
timestamp    = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
archive_name = f'mega_v2_{timestamp}.zip'
staging      = 'staging_export'
os.makedirs(staging, exist_ok=True)

for plot_f in glob.glob('plots/*.png'):
    shutil.copy(plot_f, staging); print(f'  ✓ {plot_f}')
if os.path.exists('weights'):
    shutil.copytree('weights', os.path.join(staging, 'weights'), dirs_exist_ok=True)
    print('  ✓ weights/')
if os.path.exists('summary_results.json'):
    shutil.copy('summary_results.json', staging)

shutil.make_archive(archive_name.replace('.zip', ''), 'zip', staging)
shutil.rmtree(staging, ignore_errors=True)
print(f'\n✅ Archive: {archive_name}  ({os.path.getsize(archive_name)/1e6:.1f} MB)')
try:
    display(FileLink(archive_name))
except:
    print(f'   Download: {archive_name}')


## Cell 10 — Export All Outputs

In [ ]:
# ── Export all output files to a single zip ─────────────────────────────────
# Collects: plots/, weights/, summary_results.json, the notebook itself,
# and any mega_v2_*.zip already produced by Cell 9.
import os, glob, shutil, datetime
from IPython.display import FileLink, display

timestamp    = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
export_name  = f'mega_v2_full_{timestamp}'
staging      = f'{export_name}_staging'
os.makedirs(staging, exist_ok=True)

copied = []

# Plots
for f in glob.glob('plots/*.png'):
    shutil.copy(f, staging); copied.append(f)

# Model weights
if os.path.exists('weights'):
    shutil.copytree('weights', os.path.join(staging, 'weights'), dirs_exist_ok=True)
    copied.append('weights/')

# JSON summary
if os.path.exists('summary_results.json'):
    shutil.copy('summary_results.json', staging); copied.append('summary_results.json')

# The notebook itself (Kaggle writes the live notebook to __notebook__.ipynb)
nb_src = '__notebook__.ipynb'
if os.path.exists(nb_src):
    shutil.copy(nb_src, os.path.join(staging, 'mega_v2_notebook.ipynb'))
    copied.append(nb_src)

# Any zip from Cell 9
for f in glob.glob('mega_v2_*.zip'):
    shutil.copy(f, staging); copied.append(f)

# Build the final zip
final_zip = export_name + '.zip'
shutil.make_archive(export_name, 'zip', staging)
shutil.rmtree(staging, ignore_errors=True)

print(f'\n📦 Export complete: {final_zip}  ({os.path.getsize(final_zip)/1e6:.1f} MB)')
print('   Contents:')
for item in copied:
    print(f'     ✓ {item}')

try:
    display(FileLink(final_zip))
except:
    print(f'   Download path: /kaggle/working/{final_zip}')
